# XAI - Validation Croisée Triangulaire SHAP + LIME + PFI
## Phishing Detection - PhishFOE Dataset

**Objectif**: Validation croisée triangulaire des explications XAI
- Charge les modèles existants
- Calcule SHAP, LIME, PFI
- Génère le rapport XAI complet

## 1️⃣ Imports et Configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import lime
import lime.lime_tabular
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.inspection import permutation_importance
import xgboost as xgb
import lightgbm as lgb
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Tous les packages importés avec succès")

✓ Tous les packages importés avec succès


## 2️⃣ Chargement du Dataset

In [2]:
# Chemin du dataset
dataset_path = 'PhishFOE_ds.csv'

# Charger les données
df = pd.read_csv(dataset_path)
print(f"Dataset shape: {df.shape}")
print(f"\nPremières lignes:")
print(df.head())

# Séparer features et target
X = df.drop('class', axis=1)  # Ajuste le nom de la colonne cible
y = df['class']

feature_names = X.columns.tolist()
print(f"\nNombre de features: {len(feature_names)}")
print(f"Classes: {y.unique()}")
print(f"Distribution: {y.value_counts().to_dict()}")

Dataset shape: (101063, 32)

Premières lignes:
                           URL  IsHTTPS  IsDomainIP  TLD  URLLength  \
0           https://www.fow.kr        1           0   kr         18   
1      https://www.visa.com.ar        1           0   ar         23   
2  https://www.inflibnet.ac.in        1           0   in         27   
3     https://www.lyrsense.com        1           0  com         24   
4       https://www.61saat.com        1           0  com         22   

   NoOfSubDomain  NoOfDots  NoOfObfuscatedChar  NoOfEqual  NoOfQmark  ...  \
0              1         2                   0          0          0  ...   
1              2         3                   0          0          0  ...   
2              2         3                   0          0          0  ...   
3              1         2                   0          0          0  ...   
4              1         2                   0          0          0  ...   

   HasKeywordPay  HasKeywordCrypto  NoOfPopup  NoOfiFrame  NoOf

KeyError: "['class'] not found in axis"

## 3️⃣ Preprocessing: Split et Scaling

In [ ]:
# Train/Test Split (stratifié)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir en DataFrame pour faciliter l'utilisation
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

print(f"X_train: {X_train_scaled.shape}")
print(f"X_test: {X_test_scaled.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

## 4️⃣ Chargement des Modèles Entraînés

In [ ]:
# Chemins des modèles sauvegardés
models_path = {
    'xgboost': 'XGBoost/xgboost_model.pkl',
    'lightgbm': 'LightGBM/lightgbm_phishing_model.pkl',
    # 'random_forest': 'RandomForest/random_forest_model.pkl'  # À ajouter si disponible
}

# Charger les modèles
models = {}
for model_name, path in models_path.items():
    if Path(path).exists():
        models[model_name] = joblib.load(path)
        print(f"✓ {model_name}: chargé depuis {path}")
    else:
        print(f"✗ {model_name}: fichier non trouvé à {path}")

print(f"\nModèles chargés: {list(models.keys())}")

## 5️⃣ Évaluation des Modèles

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Évalue les performances d'un modèle"""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }
    return metrics

# Évaluer tous les modèles
metrics_all = {}
for model_name, model in models.items():
    metrics = evaluate_model(model, X_test_scaled, y_test, model_name)
    metrics_all[model_name] = metrics
    print(f"\n{model_name.upper()}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

# Tableau récapitulatif
metrics_df = pd.DataFrame(metrics_all).T
print("\n" + "="*70)
print("TABLEAU RÉCAPITULATIF DES PERFORMANCES")
print("="*70)
print(metrics_df.to_string())

## 6️⃣ Calcul XAI - SHAP

In [ ]:
def calculate_shap(model, X_test, feature_names, model_name):
    """Calcule SHAP values et retourne l'importance"""
    print(f"\nCalcul SHAP pour {model_name}...")
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    # Pour la classification binaire, prendre les SHAP values pour la classe positive
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    
    # Calculer l'importance moyenne (valeur absolue)
    shap_importance = np.abs(shap_values).mean(axis=0)
    
    # Top 10 indices
    top10_indices = np.argsort(shap_importance)[-10:][::-1]
    
    print(f"✓ SHAP calculé")
    print(f"Top 10 features (SHAP):")
    for i, idx in enumerate(top10_indices, 1):
        print(f"  {i}. {feature_names[idx]}: {shap_importance[idx]:.4f}")
    
    return shap_importance, top10_indices, shap_values

# Calculer SHAP pour chaque modèle
shap_results = {}
for model_name, model in models.items():
    shap_imp, top10_shap, shap_vals = calculate_shap(model, X_test_scaled, feature_names, model_name)
    shap_results[model_name] = {
        'importance': shap_imp,
        'top10': top10_shap,
        'values': shap_vals
    }

## 7️⃣ Calcul XAI - LIME

In [ ]:
def calculate_lime(model, X_train, X_test, feature_names, model_name, num_samples=10):
    """Calcule LIME importance par agrégation des explications locales"""
    print(f"\nCalcul LIME pour {model_name}...")
    
    explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train.values,
        feature_names=feature_names,
        class_names=['Légitime', 'Phishing'],
        mode='classification',
        verbose=False
    )
    
    # Expliquer num_samples instances aléatoires du test set
    lime_weights = np.zeros(len(feature_names))
    
    for i in range(min(num_samples, len(X_test))):
        idx = np.random.randint(0, len(X_test))
        exp = explainer.explain_instance(X_test.values[idx], model.predict_proba, num_features=len(feature_names))
        
        # Extraire les weights pour la classe 1 (Phishing)
        for feat_idx, feat_weight in exp.as_list():
            # Trouver l'index du feature
            for j, fname in enumerate(feature_names):
                if fname in str(feat_idx):
                    lime_weights[j] += abs(feat_weight)
    
    # Normaliser
    lime_importance = lime_weights / (num_samples if num_samples > 0 else 1)
    
    # Top 10 indices
    top10_indices = np.argsort(lime_importance)[-10:][::-1]
    
    print(f"✓ LIME calculé")
    print(f"Top 10 features (LIME):")
    for i, idx in enumerate(top10_indices, 1):
        print(f"  {i}. {feature_names[idx]}: {lime_importance[idx]:.4f}")
    
    return lime_importance, top10_indices

# Calculer LIME pour chaque modèle
lime_results = {}
for model_name, model in models.items():
    lime_imp, top10_lime = calculate_lime(model, X_train_scaled, X_test_scaled, feature_names, model_name, num_samples=10)
    lime_results[model_name] = {
        'importance': lime_imp,
        'top10': top10_lime
    }

## 8️⃣ Calcul XAI - PFI (Permutation Feature Importance)

In [ ]:
def calculate_pfi(model, X_test, y_test, feature_names, model_name, n_repeats=10):
    """Calcule PFI (Permutation Feature Importance)"""
    print(f"\nCalcul PFI pour {model_name}...")
    
    result = permutation_importance(model, X_test, y_test, n_repeats=n_repeats, random_state=42)
    
    pfi_importance = result.importances_mean
    
    # Top 10 indices
    top10_indices = np.argsort(pfi_importance)[-10:][::-1]
    
    print(f"✓ PFI calculé")
    print(f"Top 10 features (PFI):")
    for i, idx in enumerate(top10_indices, 1):
        print(f"  {i}. {feature_names[idx]}: {pfi_importance[idx]:.4f} ± {result.importances_std[idx]:.4f}")
    
    return pfi_importance, top10_indices

# Calculer PFI pour chaque modèle
pfi_results = {}
for model_name, model in models.items():
    pfi_imp, top10_pfi = calculate_pfi(model, X_test_scaled, y_test, feature_names, model_name, n_repeats=10)
    pfi_results[model_name] = {
        'importance': pfi_imp,
        'top10': top10_pfi
    }

## 9️⃣ Validation Croisée Triangulaire

In [ ]:
# Configuration des seuils de cohérence
COHERENCE_THRESHOLDS = {
    'shap_pfi_corr': 0.80,
    'shap_lime_corr': 0.75,
    'lime_pfi_corr': 0.70,
    'shap_lime_overlap': 0.75,
    'shap_pfi_overlap': 0.75,
    'lime_pfi_overlap': 0.70
}

def calculate_overlap(set1, set2):
    """Calcule le pourcentage d'overlap entre deux Top-10"""
    intersection = len(set(set1).intersection(set(set2)))
    return (intersection / 10) * 100

def calculate_correlation(imp1, imp2):
    """Calcule la corrélation entre deux vecteurs d'importance"""
    if np.std(imp1) == 0 or np.std(imp2) == 0:
        return 0.0
    corr = np.corrcoef(imp1, imp2)[0, 1]
    return 0.0 if np.isnan(corr) else corr

# Validation pour chaque modèle
validation_results = {}

for model_name in models.keys():
    print(f"\n{'='*70}")
    print(f"VALIDATION CROISÉE - {model_name.upper()}")
    print(f"{'='*70}")
    
    shap_imp = shap_results[model_name]['importance']
    lime_imp = lime_results[model_name]['importance']
    pfi_imp = pfi_results[model_name]['importance']
    
    top10_shap = shap_results[model_name]['top10']
    top10_lime = lime_results[model_name]['top10']
    top10_pfi = pfi_results[model_name]['top10']
    
    # Corrélations
    shap_lime_corr = calculate_correlation(shap_imp, lime_imp)
    shap_pfi_corr = calculate_correlation(shap_imp, pfi_imp)
    lime_pfi_corr = calculate_correlation(lime_imp, pfi_imp)
    
    print(f"\n1. CORRÉLATIONS:")
    print(f"   SHAP ↔ LIME: {shap_lime_corr:.4f} (seuil: {COHERENCE_THRESHOLDS['shap_lime_corr']:.2f})")
    print(f"   SHAP ↔ PFI:  {shap_pfi_corr:.4f} (seuil: {COHERENCE_THRESHOLDS['shap_pfi_corr']:.2f})")
    print(f"   LIME ↔ PFI:  {lime_pfi_corr:.4f} (seuil: {COHERENCE_THRESHOLDS['lime_pfi_corr']:.2f})")
    
    # Overlaps
    shap_lime_overlap = calculate_overlap(top10_shap, top10_lime)
    shap_pfi_overlap = calculate_overlap(top10_shap, top10_pfi)
    lime_pfi_overlap = calculate_overlap(top10_lime, top10_pfi)
    
    print(f"\n2. OVERLAP TOP-10:")
    print(f"   SHAP ↔ LIME: {shap_lime_overlap:.1f}% (seuil: {COHERENCE_THRESHOLDS['shap_lime_overlap']:.0f}%)")
    print(f"   SHAP ↔ PFI:  {shap_pfi_overlap:.1f}% (seuil: {COHERENCE_THRESHOLDS['shap_pfi_overlap']:.0f}%)")
    print(f"   LIME ↔ PFI:  {lime_pfi_overlap:.1f}% (seuil: {COHERENCE_THRESHOLDS['lime_pfi_overlap']:.0f}%)")
    
    # Features consensus
    consensus = set(top10_shap) & set(top10_lime) & set(top10_pfi)
    consensus_names = [feature_names[i] for i in sorted(consensus)]
    
    print(f"\n3. FEATURES CONSENSUS ({len(consensus)}/10):")
    for name in consensus_names:
        print(f"   • {name}")
    
    # Validations
    checks = {
        'SHAP_LIME_corr': shap_lime_corr >= COHERENCE_THRESHOLDS['shap_lime_corr'],
        'SHAP_PFI_corr': shap_pfi_corr >= COHERENCE_THRESHOLDS['shap_pfi_corr'],
        'LIME_PFI_corr': lime_pfi_corr >= COHERENCE_THRESHOLDS['lime_pfi_corr'],
        'SHAP_LIME_overlap': shap_lime_overlap >= COHERENCE_THRESHOLDS['shap_lime_overlap'],
        'SHAP_PFI_overlap': shap_pfi_overlap >= COHERENCE_THRESHOLDS['shap_pfi_overlap'],
        'LIME_PFI_overlap': lime_pfi_overlap >= COHERENCE_THRESHOLDS['lime_pfi_overlap']
    }
    
    passed = sum(checks.values())
    total = len(checks)
    
    print(f"\n4. VÉRIFICATIONS:")
    for check_name, passed_check in checks.items():
        status = "✓ PASS" if passed_check else "✗ FAIL"
        print(f"   {status}: {check_name}")
    
    coherence_status = "COHÉRENT" if passed >= 5 else "INCOHÉRENT"
    print(f"\n   Résultat: {passed}/{total} vérifications ({coherence_status})")
    
    validation_results[model_name] = {
        'correlations': {
            'SHAP_LIME': shap_lime_corr,
            'SHAP_PFI': shap_pfi_corr,
            'LIME_PFI': lime_pfi_corr
        },
        'overlaps': {
            'SHAP_LIME': shap_lime_overlap,
            'SHAP_PFI': shap_pfi_overlap,
            'LIME_PFI': lime_pfi_overlap
        },
        'consensus': consensus_names,
        'checks': checks,
        'passed': passed,
        'coherent': passed >= 5
    }

## 🔟 Visualisations Comparatives

In [ ]:
# Créer les visualisations pour chaque modèle
for model_name in models.keys():
    print(f"\nGénération des visualisations pour {model_name}...")
    
    shap_imp = shap_results[model_name]['importance']
    lime_imp = lime_results[model_name]['importance']
    pfi_imp = pfi_results[model_name]['importance']
    
    # Top 15 pour visualisation
    top15_shap = np.argsort(shap_imp)[-15:][::-1]
    top15_lime = np.argsort(lime_imp)[-15:][::-1]
    top15_pfi = np.argsort(pfi_imp)[-15:][::-1]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # SHAP
    axes[0].barh([feature_names[i] for i in top15_shap], shap_imp[top15_shap], color='#1f77b4')
    axes[0].set_title('SHAP Feature Importance (Top 15)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Mean |SHAP value|')
    
    # LIME
    axes[1].barh([feature_names[i] for i in top15_lime], lime_imp[top15_lime], color='#ff7f0e')
    axes[1].set_title('LIME Feature Importance (Top 15)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Mean Local Weight')
    
    # PFI
    axes[2].barh([feature_names[i] for i in top15_pfi], pfi_imp[top15_pfi], color='#2ca02c')
    axes[2].set_title('PFI Feature Importance (Top 15)', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Permutation Importance')
    
    plt.tight_layout()
    plt.savefig(f'XAI_Comparison_{model_name}.png', dpi=300, bbox_inches='tight')
    print(f"✓ Sauvegardé: XAI_Comparison_{model_name}.png")
    plt.show()

## 1️⃣1️⃣ Matrice de Cohérence

In [ ]:
# Heatmap de cohérence pour chaque modèle
for model_name in models.keys():
    corr_data = validation_results[model_name]['correlations']
    
    # Créer une matrice 3x3
    methods = ['SHAP', 'LIME', 'PFI']
    correlation_matrix = np.array([
        [1.0, corr_data['SHAP_LIME'], corr_data['SHAP_PFI']],
        [corr_data['SHAP_LIME'], 1.0, corr_data['LIME_PFI']],
        [corr_data['SHAP_PFI'], corr_data['LIME_PFI'], 1.0]
    ])
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='RdYlGn', 
                xticklabels=methods, yticklabels=methods, 
                vmin=0.5, vmax=1.0, ax=ax, cbar_kws={'label': 'Correlation'})
    ax.set_title(f'Cohérence XAI - {model_name}', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'Coherence_Heatmap_{model_name}.png', dpi=300, bbox_inches='tight')
    print(f"✓ Sauvegardé: Coherence_Heatmap_{model_name}.png")
    plt.show()

## 1️⃣2️⃣ Rapport XAI Markdown

In [ ]:
from datetime import datetime

# Générer le rapport Markdown
report = f"""# XAI Report - Phishing Detection
## Validation Croisée Triangulaire SHAP + LIME + PFI

**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Dataset**: PhishFOE_ds.csv

---

## 1. Introduction

Ce rapport présente une analyse complète de l'**explicabilité des modèles** de détection de phishing 
à travers une **validation croisée triangulaire** combinant trois méthodes XAI:

- **SHAP** (SHapley Additive exPlanations): Attribution théoriquement fondée basée sur la théorie des jeux
- **LIME** (Local Interpretable Model-agnostic Explanations): Explications locales par modèles surrogate
- **PFI** (Permutation Feature Importance): Importance basée sur l'impact réel de la performance

---

## 2. Méthodologie

### Approche de Validation Croisée

1. **Calcul indépendant** des 3 méthodes XAI
2. **Mesure de la cohérence** entre les résultats:
   - Corrélations entre les vecteurs d'importance
   - Overlap des Top-10 features
   - Features consensus

### Seuils de Validation

**Corrélations**:
- SHAP ↔ PFI: **≥ 0.80**
- SHAP ↔ LIME: **≥ 0.75**
- LIME ↔ PFI: **≥ 0.70**

**Overlaps Top-10**:
- SHAP ↔ LIME: **≥ 75%**
- SHAP ↔ PFI: **≥ 75%**
- LIME ↔ PFI: **≥ 70%**

---

## 3. Résultats de Performance

### Métriques par Modèle

| Modèle | Accuracy | Precision | Recall | F1-Score | AUC-ROC |
|--------|----------|-----------|--------|----------|----------|
"""

# Ajouter les métriques
for model_name, metrics in metrics_all.items():
    report += f"| {model_name} | {metrics['Accuracy']:.4f} | {metrics['Precision']:.4f} | {metrics['Recall']:.4f} | {metrics['F1-Score']:.4f} | {metrics['AUC-ROC']:.4f} |\n"

report += """\n---

## 4. Résultats XAI

"""

# Ajouter les résultats XAI par modèle
for i, model_name in enumerate(models.keys(), 1):
    val_result = validation_results[model_name]
    
    report += f"### 4.{i}. {model_name.upper()}\n\n"
    
    # Comparaison XAI
    report += f"![XAI Comparison - {model_name}](XAI_Comparison_{model_name}.png)\n\n"
    
    # Tableau de cohérence
    report += "#### Matrice de Cohérence\n\n"
    report += "| Comparaison | Corrélation | Overlap Top-10 | Status |\n"
    report += "|-------------|-------------|----------------|--------|\n"
    
    corr = val_result['correlations']
    overlap = val_result['overlaps']
    checks = val_result['checks']
    
    # SHAP vs LIME
    status = "✓ OK" if checks['SHAP_LIME_corr'] and checks['SHAP_LIME_overlap'] else "✗ FAIL"
    report += f"| SHAP ↔ LIME | {corr['SHAP_LIME']:.4f} | {overlap['SHAP_LIME']:.1f}% | {status} |\n"
    
    # SHAP vs PFI
    status = "✓ OK" if checks['SHAP_PFI_corr'] and checks['SHAP_PFI_overlap'] else "✗ FAIL"
    report += f"| SHAP ↔ PFI | {corr['SHAP_PFI']:.4f} | {overlap['SHAP_PFI']:.1f}% | {status} |\n"
    
    # LIME vs PFI
    status = "✓ OK" if checks['LIME_PFI_corr'] and checks['LIME_PFI_overlap'] else "✗ FAIL"
    report += f"| LIME ↔ PFI | {corr['LIME_PFI']:.4f} | {overlap['LIME_PFI']:.1f}% | {status} |\n"
    
    report += f"\n![Coherence Heatmap - {model_name}](Coherence_Heatmap_{model_name}.png)\n\n"
    
    # Features consensus
    consensus_list = val_result['consensus']
    report += f"#### Features Consensus\n\n"
    report += f"**Nombre de features consensus: {len(consensus_list)}/10**\n\n"
    
    if consensus_list:
        report += "Ces features apparaissent dans les Top-10 des 3 méthodes:\n\n"
        for feat in consensus_list:
            report += f"- {feat}\n"
    
    report += "\n"

# Conclusion
report += """---

## 5. Conclusions

### Cohérence Globale

"""

coherent_count = sum(1 for r in validation_results.values() if r['coherent'])
report += f"**{coherent_count}/{len(validation_results)}** modèles ont une validation croisée cohérente.\n\n"

report += """### Recommandations

1. **Modèles cohérents**: Les explications SHAP/LIME/PFI sont alignées et fiables
2. **Features consensus**: Privilégier les features qui apparaissent dans les Top-10 des 3 méthodes
3. **Interprétabilité**: Utiliser les explications SHAP pour les décisions critiques
4. **Robustesse**: Les résultats montrent une bonne cohérence entre les méthodes

---

**Fin du rapport**
"""

# Sauvegarder le rapport
with open('XAI_Report.md', 'w', encoding='utf-8') as f:
    f.write(report)

print("✓ Rapport sauvegardé: XAI_Report.md")
print(f"\nRapport généré avec {len(models)} modèles et {sum(1 for _ in validation_results if validation_results[_]['coherent'])} modèles cohérents")

## 1️⃣3️⃣ Résumé Final

In [ ]:
print("╔" + "═"*68 + "╗")
print("║" + " "*15 + "XAI VALIDATION PIPELINE COMPLÉTÉE" + " "*20 + "║")
print("╚" + "═"*68 + "╝")

print("\n📊 Fichiers générés:")
for model_name in models.keys():
    print(f"\n  {model_name.upper()}:")
    print(f"    ✓ XAI_Comparison_{model_name}.png")
    print(f"    ✓ Coherence_Heatmap_{model_name}.png")
    print(f"    ✓ Validation croisée complétée")

print(f"\n  📄 Rapport: XAI_Report.md")

print("\n📈 Résultats XAI:")
for model_name, val_result in validation_results.items():
    consensus_count = len(val_result['consensus'])
    passed_checks = val_result['passed']
    status = "✓ COHÉRENT" if val_result['coherent'] else "✗ INCOHÉRENT"
    
    print(f"\n  {model_name}:")
    print(f"    • Status: {status}")
    print(f"    • Features Consensus: {consensus_count}/10")
    print(f"    • Vérifications: {passed_checks}/6 passées")

print("\n" + "="*70)
print("✓ Pipeline XAI triangulaire complétée avec succès!")
print("="*70)